# Cognition & Computation - Final Project

**Student:** Diana Cristina Andrade Damian  
**ID number:** 2141895


For this project I decided to work with the **Fashion MNIST Dataset**. This is a dataset that consists of a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes.
More information about this dataset can be found in the following link: https://www.kaggle.com/datasets/zalando-research/fashionmnist

The purpose of this project is to explore Deep Belief Networks' (DBNs) performance, understand hierarchical models' behaviour at each level, and compare it with different architectures on challenging situations, such as noisy datasets and adversarial attacks.

The main architecture implemented in this project is based on the Hinton and Salakhutdinov's (2006). This architecture is composed by a stack of three Restricted Boltzman Machines (RBMs), more details about the architecture will be described later.

During the development of this project, we will start first by training our DBN model in an unsupervised way. Posterior to that we will explore its weights on each layer and hidden representations, that later will be used to train it in a supervised way. Finally a comparison with different models will be implemented.

# 1. DEEP BELIEF NETWORK

First step: download two simple scripts implementing a Deep Belief Network in PyTorch and import a few Python libraries.

In [ ]:
def get_dbn_library():
  files = ["DBN.py", "RBM.py"]
  repository_url = "https://raw.githubusercontent.com/flavio2018/Deep-Belief-Network-pytorch/master/"
  for file in files:
    ! wget -O {file} {repository_url}{file}

In [ ]:
%%capture
get_dbn_library()

In [ ]:
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
import scipy.cluster as cluster
import sklearn.preprocessing
import torch
import torchvision as tv
import time
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split


from DBN import DBN

We can choose dynamically the kind of hardware device used for computations (CPU or GPU).

In [ ]:
print(torch.cuda.is_available())
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

Then, let's download the Fashion MNIST dataset. The dataset corresponds to a csv file, so we can download it as a dataframe using pandas. We download both the train and test datasets (we will use the test set later on). It's important to highlight the structure of the data, where each row represents a image, while the first column corresponds to the label (10 possible options) and the following ones to the pixels. For this reason we have a 28x28 + 1 columns = 785.

First, we create a copy of the original dataframe and extract the labels and pixels into two different variables. Then, we divide each pixel by the maximum value (255 in RGB notation) to get input values encoded between 0 and 1. Once data is normalized, the training data is split into training and validation sets, where the training and validations sets contain 50,400 and 9,600 images, respectively.

Finally, we transform the data from a numpy array, into torch type.
If using CUDA, we also need to transfer the data to the GPU.

Note: Same procedure is applied for training and testing data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

#Downloads the train set
FashionMNIST_train = pd.read_csv(path + "/fashion-mnist_train.csv")

#Downloads the test set
FashionMNIST_test = pd.read_csv(path + "/fashion-mnist_test.csv")

In [ ]:
#Creates copy of dataframes
copy_train = FashionMNIST_train.copy()
copy_test = FashionMNIST_test.copy()

#Selects the labels for each dataset and turns them into a numpy type
train_labels = copy_train["label"].to_numpy()
test_labels = copy_test["label"].to_numpy()

#Selects the pixels for each dataset, turning them into a numpy type
#And normalizes them by diving them by 255
train_pixels_all = copy_train.iloc[:, 1:].to_numpy() / 255
test_pixels = copy_test.iloc[:, 1:].to_numpy() /255

#Splits the training dataset into training and validation
train_pixels, val_pixels, train_labels, val_labels = train_test_split(train_pixels_all, train_labels, test_size=0.16, random_state=42)

#Now, turns data into a torch type
train_pixels = torch.from_numpy(train_pixels)
val_pixels = torch.from_numpy(val_pixels)
test_pixels = torch.from_numpy(test_pixels)
train_labels = torch.from_numpy(train_labels)
val_labels = torch.from_numpy(val_labels)
test_labels = torch.from_numpy(test_labels)

#Sends data to the GPU
train_pixels = train_pixels.to(device)
val_pixels = val_pixels.to(device)
test_pixels = test_pixels.to(device)
train_labels = train_labels.to(device)
val_labels = val_labels.to(device)
test_labels = test_labels.to(device)

Let's visualize one training image:

In [ ]:
idx = 3
img = train_pixels[idx].cpu()  # NB: to plot the data, we need to move it back from the GPU memory!
print("The image shows the class: {}".format(train_labels[idx]))
plt.imshow(img.reshape(28, 28) , cmap = 'gray') # Reshape img to 28x28
plt.show()

Each training and test example is assigned to one of the following labels:

0: T-shirt/top

1: Trouser

2: Pullover

3: Dress

4: Coat

5: Sandal

6: Shirt

7: Sneaker

8: Bag

9: Ankle boot

As we can appreciate from the previous image, the element corresponds to a sandal, which is encoded as class number 5.

# Training a Deep Belief Network

First, let's recall the definition of a DBN. A DBN is a deep learning architecture formed by stacked Restricted Boltzman Machines (RBM). These RBM are networks of undirected graphical models, which take their name because they restrict connections between units corresponding to the same layer, and they allow only visible and hidden units' connections.

The model is composed by a stack of 3 RBMs. These are neural networks where the probability of the unit's configuration is determined by an energy function, in our case Cross Entropy.

Model is initially trained in a unsupervised way, meaning that the model is not in contact with the labeled data. In this way, the model will be forced to learn hidden patterns and clusters inside the data. Our learning algorithm is Contrastive Divergence, which is a type of contrastive optimization where the gradient of the log-likelihood is approximated. Essentially is divided in two phases: positive and negative. During the positive phase the visible units of the training set are campled, forcing them to keep a constant value. Subsequently, the negative phase begings, where a Markov Chain Monte Carlo (MCMC) algorithm is run to obtain the model's reconstruction of that instance.

In our model, k refers to the number of iterations of the contrastive divergence learning. This is one of the many parameters we can adjust to improve our model's performance.

During this phase is important to make sure that the reconstruction error is decreasing on each layer. This term will provide a general overview of the model's performance. In this case, the error is decreasing in all the layers, so we can continue with the training.


In [ ]:
num_epochs = 80
batch_size = 125

fashion_dbn_mnist = DBN(visible_units=28*28,          # dimensionality of the sensory data
                    hidden_units=[700, 500, 300], # size of hidden layers
                    k=1,                          # reconstruction steps in Contrastive Divergence
                    learning_rate=0.1,
                    learning_rate_decay=False, #if true lr gets smaller as training proceeds
                    initial_momentum=0.5,
                    final_momentum=0.95,
                    weight_decay=0.0001,
                    xavier_init=False,
                    increase_to_cd_k=False,
                    use_gpu=torch.cuda.is_available())

fashion_dbn_mnist.train_static(
    train_pixels,
    train_labels,
    num_epochs,
    batch_size
)

Reducing the number of hidden units in the last layers don't affect the recovery error. Since deeper layers learn higher-level, more abstract features, they need in fact fewer units

## Visualizing receptive fields

Once the training is done, we can now plot our learned weights, to visualize the "receptive fields". These can be defined as the size of the region in the input producing a specific feature, which are basically telling us which parts and features of the image trigger the unit associated with a specific vector.

The following are auxiliar functions that help us to obtain the weights for an specific layer, apply a threshold and a scaler to process it, and finally plot them to visualize them.

In [ ]:
def get_weights(dbn, layer):
  return dbn.rbm_layers[layer].W.cpu().numpy()

def apply_threshold(weights, threshold=0):
  return weights * (abs(weights) > threshold)

def apply_min_max_scaler(learned_weights):
  original_shape = learned_weights.shape
  min_max_scaler = sklearn.preprocessing.MinMaxScaler()
  min_max_scaled_learned_weights = min_max_scaler.fit_transform(learned_weights.ravel().reshape(-1,1))
  min_max_scaled_learned_weights = min_max_scaled_learned_weights.reshape(original_shape)
  return min_max_scaled_learned_weights

def plot_layer_receptive_fields(weights, layer):
  num_subplots = 100
  n_rows_cols = int(math.sqrt(num_subplots))
  fig, axes = plt.subplots(n_rows_cols, n_rows_cols, sharex=True, sharey=True, figsize=(10, 10))
  fig.suptitle(f'{layer}-layer weight visualization')
  for i in range(num_subplots):
    row = i % n_rows_cols
    col = i // n_rows_cols
    axes[row, col].imshow(weights[i,:].reshape((28,28)), cmap=plt.cm.gray)  # here we select the weights we want to plot

In [ ]:
#First, we obtained the weights of each layer
w1_f = get_weights(fashion_dbn_mnist, layer=0)
w2_f = get_weights(fashion_dbn_mnist, layer=1)
w3_f = get_weights(fashion_dbn_mnist, layer=2)

print("First layer shape: ", w1_f.shape)
print("Second layer shape: ", w2_f.shape)
print("Third layer shape: ", w3_f.shape)

As it is observed, the weights in the second and third hidden layers don't have the same dimensionality as Fashion MNIST, therefore, we will need to project each of the vectors in a space of dimensionality `784` (`28`x`28`) in order to visualize them as images. The projection can be done in many ways, here for simplicity we consider a [linear projection](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html).

In [ ]:
#A 0.1 threshold is applied
w1_f = apply_threshold(w1_f, 0.1)
w2_f = apply_threshold(w2_f, 0.1)
w3_f = apply_threshold(w3_f, 0.1)

#Matrix multiplication between layer's weights, to perform the projection for 2nd and 3rd layers
w_product_12_f = (w1_f @ w2_f)  # here we do the projection
w_product_23_f = (w_product_12_f @ w3_f)  # here we do the projection

#Same threshold is applied, now on the matrix products
w_product_12_f = apply_threshold(w_product_12_f, 0.1)
w_product_23_f = apply_threshold(w_product_23_f, 0.1)

#Scaler
w1_f = apply_min_max_scaler(w1_f)
w_product_12_f = apply_min_max_scaler(w_product_12_f)
w_product_23_f = apply_min_max_scaler(w_product_23_f)

#Plotting of the weights
plot_layer_receptive_fields(w1_f.T, 'First')
plot_layer_receptive_fields(w_product_12_f.T, 'Second')
plot_layer_receptive_fields(w_product_23_f.T, 'Third')

Reducing the number of hidden units for the last layer helped to reduce the number of dead units, which are identified as totally black elements. These "dead" elements are basically telling us they had learned nothing, and consequently do not contribute to the model's performance. In fact, they only provoke a waste of memory and efficacy decline.  

One hyphotesis about the cause of dead neurons is related to the redundancy, considering that deeper layers learn more abstract features by combining simpler features, we can agree in fact that a lower number of units are needed.

## Clustering internal representations

We can also examine the properties of the learned *distributed* representations. For example, we can compute the centroid of the representations learned for each class, and see how close they are to each other using a standard hierarchical clustering algorithm.

This implementation of the `DBN` contains internally several `RBM` objects. Therefore, we will need to compute the hidden representation using the weights of each `RBM`.

In [ ]:
#Function to get the k-th layer
def get_kth_layer_repr(input, k, device, model):
  flattened_input = input.view((input.shape[0], -1)).type(torch.FloatTensor).to(device)
  hidden_repr, __ = model.rbm_layers[k].to_hidden(flattened_input)  # here we access the RBM object
  return hidden_repr

#Functions to compute the centroid (mean) of the representations
def get_mask(label):  # we use this function to filter by class
  labels = train_labels.cpu().numpy()
  return labels == label

def get_label_to_mean_hidd_repr(hidden_repr):
  hidden_repr_np = hidden_repr.cpu().numpy()
  return {
    label: hidden_repr_np[get_mask(label)].mean(axis=0)  # here we filter by class and compute the mean
    for label in range(10)
  }

def get_hidden_reprs_matrix(hidden_repr):  # we use this to build the matrices
  label_to_mean_hidd_repr = get_label_to_mean_hidd_repr(hidden_repr)
  return np.concatenate(
    [np.expand_dims(label_to_mean_hidd_repr[label], axis=0)  # here we adjust the shape of centroids to do the concat
    for label in range(10)])

#Function to visualize the clustering algorithm in a dendogram plot
def plot_dendrogram(mean_repr_matrix, title=""):
  fig, ax = plt.subplots()
  linkage = cluster.hierarchy.linkage(mean_repr_matrix, method="complete")  # we run the clustering algorithm here
  dendrogram = cluster.hierarchy.dendrogram(linkage)
  ax.set_title(title)

The representations computed for the second hidden layer are derived using the ones of the first hidden layer, and so on until we get to the deepest layer. Since now we have a training and validation set, it's necessary to obtain the hidden representations of the validation set as well, since we'll need them during the supervised training of the model.

In [ ]:
#Hidden representation for training data
hidden_repr_1_f = get_kth_layer_repr(train_pixels.data, 0, device, fashion_dbn_mnist)
hidden_repr_2_f = get_kth_layer_repr(hidden_repr_1_f, 1, device, fashion_dbn_mnist)
hidden_repr_3_f = get_kth_layer_repr(hidden_repr_2_f, 2, device, fashion_dbn_mnist)
print("TRAINING SET")
print("First layer hidden representation shape: ", hidden_repr_1_f.shape)
print("Second layer hidden representation shape: ", hidden_repr_2_f.shape)
print("Third layer hidden representation shape: ", hidden_repr_3_f.shape)

print("VALIDATION SET")
#Hidden representation for validation data
hidden_repr_1_valf = get_kth_layer_repr(val_pixels.data, 0, device, fashion_dbn_mnist)
hidden_repr_2_valf = get_kth_layer_repr(hidden_repr_1_valf, 1, device, fashion_dbn_mnist)
hidden_repr_3_valf = get_kth_layer_repr(hidden_repr_2_valf, 2, device, fashion_dbn_mnist)
print("First layer hidden representation shape: ", hidden_repr_1_valf.shape)
print("Second layer hidden representation shape: ", hidden_repr_2_valf.shape)
print("Third layer hidden representation shape: ", hidden_repr_3_valf.shape)

Finally, we compute the centroid (mean) of the representations of each class and we build a matrix containing all the centroids to comply with the input required by the clustering algorithm. Now we can run the clustering algorithm and visualize it output in a dendrogram plot.

In [ ]:
mean_hidd_repr_matrix_3f = get_hidden_reprs_matrix(hidden_repr_3_f)
plot_dendrogram(mean_hidd_repr_matrix_3f, "Third hidden layer")

When interpreting a dendogram is important to consider that clusters merging at lower heights are more alike, while those merging at higher heights are less similar.
Dendrograms summarize underline distance matrix.

From our dendogram we can appreciate three well defined clusters. The red one in fact can be associated with upper body clothing, containing the most similar classes, corresponding to number 2 (pullover) and 4 (coat), and other classes such as 6 (shirt), 8 (bag) and 0 (t-shirt/top). In this case the bag is the only element that does not truly correspond to the type of clothing, it would be interesting to try different architectures to analyze the classification of this element.

On the orange cluster we can find the classes 5 (sandal) and 7 (sneaker) being the most similar ones, followed by 9 (ankle boot). It's evident that this cluster refers to shoes.

Finally, the green cluster contains the class 1 (trouser) and 3 (dress).

##Supervised training

The second phase of the training is now performed in a supervised way. This phase is useful to improve the model's performance and create a better discriminative learning.

In order to do it, we train the linear classifiers on the hidden representations from each layer using the actual labels of the Fashion MNIST dataset as targets. In the same way, the predictions from the validation set are obtained and its accuracy is computed.

The plot of the training and validation losses help us to understand the progress of the training process and improve the hyperparameters.  

## Linear read-out

In order to implement our linear classifier, we need to decode our representations first. This process will be done by implementing a technique called "linear readout", which refers to a method of interpreting data as a simple weighted sum. Similar to the clustering technique, this can be done at each layer of the model.

The hidden representations will be extracted of the data by propagating the neurons' activations from the sensory (visible) layer in a bottom-up fashion. Finally these hidden representations will be used to classify the Fashion MNIST dataset using a simple linear model. In this way, we can assess how much information is contained on each layer.  

In [ ]:
class LinearModel(torch.nn.Module):
  def __init__(self, layer_size):
    super().__init__()
    self.linear = torch.nn.Linear(layer_size, 10)

  def forward(self, x):
    return self.linear(x)

Since each of our layer contains different amount of units, it's necessary to apply the linear model based on this information. The easier way to keep track of this it is to instantiate a linear classifier for each hidden layer of the DBN:

In [ ]:
layer_size_f = fashion_dbn_mnist.rbm_layers[0].W.shape[1]
print(layer_size_f)
linear1_f = LinearModel(layer_size_f).to(device)

layer_size_f = fashion_dbn_mnist.rbm_layers[1].W.shape[1]
print(layer_size_f)
linear2_f = LinearModel(layer_size_f).to(device)

layer_size_f = fashion_dbn_mnist.rbm_layers[2].W.shape[1]
print(layer_size_f)
linear3_f = LinearModel(layer_size_f).to(device)

In [ ]:
def train_linear(linear, train_hidden_reprs, val_hidden_reprs, epochs, early_stop=False):
  optimizer = torch.optim.Adam(linear.parameters(), lr=0.001)
  loss_fn = torch.nn.CrossEntropyLoss()
  patience = 50           # stop if no improvement for 50 epochs
  best_val_loss = float('inf')
  patience_counter = 0
  model_name = "best_model.pt"
  best_weights = None # Initialize best_weights outside the loop

  train_losses = []
  val_losses = []

  for epoch in range(epochs):
    start_time = time.time()
    #TRAINING
    optimizer.zero_grad()
    train_preds = linear(train_hidden_reprs).squeeze()
    train_targets = train_labels.reshape(train_preds.shape[0])  # here are the labels
    train_loss = loss_fn(train_preds, train_targets)
    train_loss.backward()
    optimizer.step()

    train_losses.append(train_loss.item())

    #VALIDATION
    with torch.no_grad():
      val_preds = linear(val_hidden_reprs).squeeze()
      val_targets = val_labels.reshape(val_preds.shape[0])
      val_loss = loss_fn(val_preds, val_targets)

      val_losses.append(val_loss.item())
      val_accuracy = (val_preds.argmax(dim=1) == val_targets).float().mean().item()

    end_time = time.time()

    if epoch % 200 == 0:
      print("epoch : {:3d}/{}, Training loss = {:.4f}, Validation loss={:.4f}, Validation accuracy={:.4f}".format(epoch + 1, epochs, train_loss, val_loss, val_accuracy))

    if early_stop:
      #EARLY STOPPING should be inside the loop
      if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        patience_counter = 0
        best_weights = linear.state_dict() # Update best_weights here
      else:
        patience_counter += 1

      if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch + 1}") # +1 to show actual epoch number
        linear.load_state_dict(best_weights) #Loading the best weights into the model for testing
        break

  # ---- PLOT AFTER TRAINING ----
  plt.figure(figsize=(8,5))
  plt.plot(train_losses, label="Training Loss")
  plt.plot(val_losses, label="Validation Loss")
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.title("Training vs Validation Loss")
  plt.legend()
  plt.grid(True)
  plt.show()

We will now start the supervised training of our model. In this case the validation set is a way to test our model without using the test set, so it's a great reference of the actual learning of the model.

We will set up a high number of epochs since we our model has the early stop function, in this case we make sure that we are using the best weights for the testing and we don't keep training unnecesarily.

In [ ]:
print("Implementing supervised training on each layer of the model")
print("Layer #1")
train_linear(linear1_f, hidden_repr_1_f, hidden_repr_1_valf, 3000, True)
print("Layer #2")
train_linear(linear2_f, hidden_repr_2_f, hidden_repr_2_valf, 3000, True)
print("Layer #3")
train_linear(linear3_f, hidden_repr_3_f, hidden_repr_3_valf, 3000, True)

Let's now evaluate the trained linear readouts using the hidden representations computed on the *test* set:

In [ ]:
#Hidden representations on test set for fashion_dbn_mnist
hidden_repr_1_test_f = get_kth_layer_repr(test_pixels, 0, device, fashion_dbn_mnist)
hidden_repr_2_test_f = get_kth_layer_repr(hidden_repr_1_test_f, 1, device, fashion_dbn_mnist)
hidden_repr_3_test_f = get_kth_layer_repr(hidden_repr_2_test_f, 2, device, fashion_dbn_mnist)

# compute the classifier predictions:
predictions_test1_f = linear1_f(hidden_repr_1_test_f)
predictions_test2_f = linear2_f(hidden_repr_2_test_f)
predictions_test3_f = linear3_f(hidden_repr_3_test_f)

Finally, let's compute the accuracy scores:

In [ ]:
def compute_accuracy(predictions_test, targets):
  predictions_indices = predictions_test.max(axis=1).indices  # convert probabilities to indices
  accuracy = (predictions_indices == targets).sum() / len(targets)
  return accuracy.item()

In [ ]:
#Layers 700 500 300 80 epochs + 3000
print("Accuracy scores")
print("1st test predictions: ", compute_accuracy(predictions_test1_f, test_labels))
print("2nd test predictions: ", compute_accuracy(predictions_test2_f, test_labels))
print("3rd test predictions: ", compute_accuracy(predictions_test3_f, test_labels))

In [ ]:
"""
#Layers 700 500 300 80 epochs + 3000 (SELECTED MODEL)
print("Accuracy scores")
print("1st test predictions: ", compute_accuracy(predictions_test1_f, test_labels))
print("2nd test predictions: ", compute_accuracy(predictions_test2_f, test_labels))
print("3rd test predictions: ", compute_accuracy(predictions_test3_f, test_labels))
"""

In [ ]:
"""
#Layers 600 400 200 80 epochs + 3000
print("Accuracy scores")
print("1st test predictions: ", compute_accuracy(predictions_test1_f, test_labels))
print("2nd test predictions: ", compute_accuracy(predictions_test2_f, test_labels))
print("3rd test predictions: ", compute_accuracy(predictions_test3_f, test_labels))
"""

# 2. COMPARISON WITH A FEED-FORWARD NEURAL NETWORK

Let's now train a simple feed-forward neural network with the same structure of the DBN, in order to compare a non-linear model that is trained end-to-end to solve a classification task with a simple linear classifier that solves the same task using representations of input data learned in an unsupervised way through the DBN.

In [ ]:
class Feedforward(torch.nn.Module):
  def __init__(self, first_hidden_layer_size, second_hidden_layer_size, third_hidden_layer_size):
    super().__init__()
    self.first_hidden = torch.nn.Linear(784, first_hidden_layer_size)
    self.second_hidden = torch.nn.Linear(first_hidden_layer_size, second_hidden_layer_size)
    self.third_hidden = torch.nn.Linear(second_hidden_layer_size, third_hidden_layer_size)
    self.output = torch.nn.Linear(third_hidden_layer_size, 10)

  def forward(self, input):
    relu = torch.nn.ReLU()
    first_hidden_repr = relu(self.first_hidden(input))
    second_hidden_repr = relu(self.second_hidden(first_hidden_repr))
    third_hidden_repr = relu(self.third_hidden(second_hidden_repr))
    output = self.output(third_hidden_repr)
    return output

In order to make a fair comparison, we will define the model with the same number of units per layer as the DBN and we will train it for the same number of epochs.

In [ ]:
ffnn = Feedforward(700, 500, 300).to(device)

In [ ]:
train_linear(ffnn, train_pixels.float(), val_pixels.float(), 3080, False) #3000 during the train_supervised + 80 on the unsupervised training

#we are training them for the exact same number of

In [ ]:
predictions_ffnn = ffnn(test_pixels.float())
compute_accuracy(predictions_ffnn, test_labels)

## Robustness to noise

We will now inject some noise in the input images and see how much the representations learned by the DBN and the feed-forward network are robust to perturbations in the sensory signal.

Similarly to what happens in psychophysical experiments, this will allow to create a psychometric curve describing the decrease in classification accuracy with respect to the noise level.

In [ ]:
def inject_noise(mnist_data, noise_level):
  noise = torch.randn(mnist_data.shape, device=device)*noise_level
  return mnist_data + noise

  ##It could be important to clip the values after adding noise (anything >1 turns to 1)
  ### TASK: create a very simple function that adds some Gaussian noise (see torch.randn function) to the MNIST data


Let's see what a noisy image looks like, let's inject differente levels of noise to appreciate the distortion of the image on each step. The first image corresponds to the original element, and the last one to the image with a 0.3 level of noise.

In [ ]:
num_subplots = 4
n_rows_cols = int(math.sqrt(num_subplots))
fig, axes = plt.subplots(1, 4, sharex=True, sharey=True, figsize=(10, 10))
noise_level = 0
fig.suptitle('Noise level from 0 to 0.3, step size=0.1', y=0.6)
for i in range(num_subplots):
  col = i
  mnist_test_with_noise = inject_noise(train_pixels, noise_level)
  axes[col].imshow(mnist_test_with_noise[idx].reshape(28, 28).to("cpu"), cmap="gray")
  noise_level += .1

We will now compute the hidden representations for the noisy images using the DBN. Then, we will use the read-out classifiers that we trained on the representations without noise to classify the noisy stimulus.

In [ ]:
def get_accuracy_values_at_noise_level(noise_level, plot=False):

  mnist_test_with_noise = inject_noise(test_pixels.float(), noise_level)  # first, let's create noisy test images

  hidden_repr_1_noisy = get_kth_layer_repr(mnist_test_with_noise, 0, device, fashion_dbn_mnist)  # here we compute the DBN representations
  hidden_repr_2_noisy = get_kth_layer_repr(hidden_repr_1_noisy, 1, device, fashion_dbn_mnist)
  hidden_repr_3_noisy = get_kth_layer_repr(hidden_repr_2_noisy, 2, device, fashion_dbn_mnist)

  predictions_first_hidden_noisy = linear1_f(hidden_repr_1_noisy)  # here we use the previously-trained read-out classifiers
  predictions_second_hidden_noisy = linear2_f(hidden_repr_2_noisy)
  predictions_third_hidden_noisy = linear3_f(hidden_repr_3_noisy)

  accuracy_first_hidden = compute_accuracy(predictions_first_hidden_noisy, test_labels. float())
  accuracy_second_hidden = compute_accuracy(predictions_second_hidden_noisy, test_labels.float())
  accuracy_third_hidden = compute_accuracy(predictions_third_hidden_noisy, test_labels.float())


  predictions_ffnn_noisy = ffnn(mnist_test_with_noise.data.reshape((10000, 784)))
  accuracy_ffnn = compute_accuracy(predictions_ffnn_noisy, test_labels)

  if plot==True:
    #Confusion matrices
    disp = ConfusionMatrixDisplay.from_predictions(
      test_labels.cpu(),
      predictions_first_hidden_noisy.cpu().argmax(dim=1),
      cmap=plt.cm.Blues,
      colorbar=True)
    disp.ax_.set_title('Confusion Matrix from 1st hidden representation')
    plt.show()

    disp = ConfusionMatrixDisplay.from_predictions(
      test_labels.cpu(),
      predictions_second_hidden_noisy.cpu().argmax(dim=1),
      cmap=plt.cm.Blues,
      colorbar=True)
    disp.ax_.set_title('Confusion Matrix from 2n hidden representation')
    plt.show()

    disp = ConfusionMatrixDisplay.from_predictions(
      test_labels.cpu(),
      predictions_third_hidden_noisy.cpu().argmax(dim=1),
      cmap=plt.cm.Blues,
      colorbar=True)
    disp.ax_.set_title('Confusion Matrix from 3rd hidden representation')
    plt.show()

    disp = ConfusionMatrixDisplay.from_predictions(
      test_labels.cpu(),
      predictions_ffnn_noisy.cpu().argmax(dim=1),
      cmap=plt.cm.Blues,
      colorbar=True)
    disp.ax_.set_title('Confusion Matrix from FFNN')
    plt.show()

  return accuracy_first_hidden, accuracy_second_hidden, accuracy_third_hidden, accuracy_ffnn

In [ ]:
acc = get_accuracy_values_at_noise_level(0.3, plot=True);
print("Accuracy of H1 read-out: %.3f" % acc[0])
print("Accuracy of H2 read-out: %.3f" % acc[1])
print("Accuracy of H3 read-out: %.3f" % acc[2])
print("Accuracy of FF network : %.3f" % acc[3])

From the accuracy results is evident that the FF network provides quite poor results (44.7%), in comparison with any of the three layers of the DBN, which kept considerably high results (82-85%) for a dataset affected by a 0.3 level of noise. These results are also appreciated by the confusion matrix, let's recall that a perfect confusion matrix should only have values in the diagonal, and all the rest should be filled with zeros. In this case, the results are not perfect for the DBN, but there's still a strong diagonal in any of the three layers. On the other hand, the FFNN had strong failures in the classification, for the case of the 5th label, it predicted incorrectly more than half of the data points.

Let's create the psychometric curves for the DBN (at different levels of internal representations) and for the feed-forward network. These curves can help us to determine how much our model "resisted" to the effect of the noise.

In [ ]:
def plot_noise_robustness_curves(noise_levels):
  accuracy_values_first_hidden = []
  accuracy_values_second_hidden = []
  accuracy_values_third_hidden = []
  accuracy_values_ffnn = []

  for noise_level in noise_levels:
    acc = get_accuracy_values_at_noise_level(noise_level)
    accuracy_values_first_hidden.append(acc[0])
    accuracy_values_second_hidden.append(acc[1])
    accuracy_values_third_hidden.append(acc[2])
    accuracy_values_ffnn.append(acc[3])

  fig, ax = plt.subplots()
  ax.plot(range(len(noise_levels)), accuracy_values_first_hidden)
  ax.plot(range(len(noise_levels)), accuracy_values_second_hidden)
  ax.plot(range(len(noise_levels)), accuracy_values_third_hidden)
  ax.plot(range(len(noise_levels)), accuracy_values_ffnn)

  ax.set_title("Robustness to noise")
  ax.set_xlabel("Noise level (%)")
  ax.set_ylabel("Accuracy")
  plt.xticks(range(len(noise_levels)), [int(l*100) for l in noise_levels])
  plt.legend(["First hidden", "Second hidden", "Third hidden", "FFNN"])

In [ ]:
noise_levels = np.linspace(0,2,10)
plot_noise_robustness_curves(noise_levels)

From this graph it's evident that the three layers of the DBN outstood the performance of the FF network. The accuracy of the DBN stayed higher for all the three layers with respect to the FF network at any level of noise. According to this evidence, we can confirm the robustness of DBN to external perturbations of the dataset, such as Gaussian noise.

On the other hand, when analyzing the behaviour of each layer, we can observe how the third layer had the most robust performance, presenting the higher accuracy for the full experiment. We can explain this result by considering that deeper layers learn more "abstract" features, and for this reason they are slightly less affected by perturbations.

# 3. PERTURBING THE MODELS WITH ADVERSARIAL ATTACKS

In general, with adversarial attacks we try to modify the input so that the model cannot correctly classify it anymore. This means that the loss for that specific input has to increase.

When we are training the model, we modify the model's weights based on the value of the gradient of the loss function, using the opposite direction w.r.t. the gradient because we want the loss to decrease. To create an adversarial sample we change two things in this procedure:

1. We modify the input instead of the model's weights
2. We move in the same direction as the gradient, since we want the loss function to increase.

In [ ]:
def fgsm_attack(image, epsilon, data_grad):
    # Collect the element-wise sign of the data gradient
    sign_data_grad = data_grad.sign()

    # Perturbed image by adjusting each pixel of the input image
    perturbed_image = image + epsilon*sign_data_grad

    # Adding clipping to maintain [0,1] range
    perturbed_image = torch.clamp(perturbed_image, 0, 1)

    # Return the perturbed image
    return perturbed_image

We need to define a unified architecture incorporating the DBN + readout layers, which allows to compute the gradient of the loss (classification task) with respect to the input data that is being processed.

In [ ]:
class DBNWithReadOut(torch.nn.Module):
    def __init__(self, fashion_dbn_mnist, readout):
        super().__init__()
        self.readout = readout
        self.readout_level = 2
        self.fashion_dbn_mnist = fashion_dbn_mnist
        self._require_grad()

    def _require_grad(self):
      for rbm in self.fashion_dbn_mnist.rbm_layers:
        rbm.W.requires_grad_()
        rbm.h_bias.requires_grad_()

    def forward(self, image):
      """This forward pass uses probabilities instead of samples as RBM activations
       to backpropagate the gradient"""
      p_v = image
      hidden_states = []
      for rbm in self.fashion_dbn_mnist.rbm_layers:
        #p_v = p_v.view((p_v.shape[0], -1))  # flatten
        p_v, v = rbm(p_v)  #we propagate it using rbm which is a function we acquired at the beginning of the code
        hidden_states.append(p_v)
      return self.readout.forward(hidden_states[self.readout_level])

In [ ]:
dbn_with_readout = DBNWithReadOut(fashion_dbn_mnist, linear3_f)

Let's see what an adversiarial sample looks like. Let't take one sample from the test set:

In [ ]:
test_sample_idx = 15
test_image = test_pixels[test_sample_idx]
print("The image shows the class: {} (Sandal)".format(test_labels[test_sample_idx]))
__ = plt.imshow(test_image.reshape(28,28).to('cpu'), cmap = 'gray')

Let's classify this "clean" image using both of the models we previously trained and then modify the image to attack the network. The first attacked model will correspond to the DBN with readout and the second one to the FFNN

In [ ]:
attacked_model_1 = dbn_with_readout
attacked_model_2 = ffnn

In [ ]:
test_image.requires_grad_()
model_outputs_1 = attacked_model_1(test_image.float()) # Cast to float32
prediction = torch.argmax(model_outputs_1)
print("DBN WITH READOUT")
print(f"Pedriction for this clean sample is {prediction}.")

model_outputs_2 = attacked_model_2(test_image.float()) # Cast to float32
prediction = torch.argmax(model_outputs_2)
print("FFNN")
print(f"Predition for this clean sample is {prediction}.")

As expected, both of the models predicted the label properly. Now, let's create and visualize the corresponding adversarial sample. The function loss.backward() computes the gradient for every parameter that was activated using the call requires_grad=True.

We are trying to go in the direction of the gradient to fool our model, let's see how the models respond to this.

In [ ]:
epsilon = 0.15  # define strenght of the attack, the higher this number the more difficult will be for the model to classify it
test_image_label = test_labels[test_sample_idx] # get ground truth label for that image
loss_value = torch.nn.functional.cross_entropy(model_outputs_1, test_image_label)  # get loss value
attacked_model_1.zero_grad()
loss_value.backward(retain_graph=True)
image_grad = test_image.grad.data  # get the gradient of the pixels w.r.t. the loss

perturbed_image = fgsm_attack(test_image, epsilon, image_grad)
perturbed_image_np = perturbed_image.detach().to('cpu').numpy()
__ = plt.imshow(perturbed_image_np.reshape(28,28), cmap = 'gray')

In [ ]:
model_outputs_1 = attacked_model_1(perturbed_image.float())
print("DBN WITH READOUT")
print(f"Prediction for the perturbed sample is {torch.argmax(model_outputs_1)}.")

model_outputs_2 = attacked_model_2(perturbed_image.float())
print("FFNN")
print(f"Prediction for the perturbed sample is {torch.argmax(model_outputs_2)}.")

In this example, both of our models failed to resist to the attack, and they both identified the "sandal", identified with number 5, as a "shirt", corresponding to number 6. With this experiment, we can now be sure that both of our models are vulnerable to this type of attacks. So, now let's try to teach our model how to "resist" to these tricks.

## Resisting to adversarial attacks

Let's now compare the ability to resist to adversarial attacks of our two models: the feedforward network and the DBN.

We will also test the ability of the DBN to reduce the impact of the attack by performing one "top-down" reconstruction step, from the hidden representation of the last layer to the visible units, and back to the hidden representation.


In [ ]:
def test_robustness_to_attack(model, device, test_loader, epsilon, num_steps=0, verbose=True):
    correct = 0  # count number of correct classifications
    print_reconstruction = num_steps > 0  # if we request for top-down reconstruction, show the resulting image

    for data, target in tqdm(test_loader):
        data, target = data.to(device), target.to(device)
        #data = data.reshape(-1, 784).float()
        data = data.float()
        data.requires_grad = True  # we need to get the gradient to perform the attack

        output = model.forward(data)  # forward pass through the model

        init_pred = torch.argmax(output)

        if (print_reconstruction and verbose):
          print("\nHere's the original sample:\n")
          plt.imshow(data[0].detach().to('cpu').numpy().reshape(28,28))
          plt.show()

        loss_value = torch.nn.functional.cross_entropy(output, target) # get loss value
        model.zero_grad()
        loss_value.backward()
        data_grad = data.grad.data  # collect the gradient with respect to the input data
        perturbed_data = fgsm_attack(data, epsilon, data_grad)  # call the attack function previously defined

        if (print_reconstruction and verbose):
            print("\nHere's a perturbed sample:\n")
            plt.imshow(perturbed_data[0].detach().to('cpu').numpy().reshape(28,28))
            plt.show()

        #Our defense against the adversarial attack
        # If requested, reconstruct the input iterating bottom-up and top-down sampling
        if num_steps > 0:
            for __ in range(0, num_steps):
                perturbed_data, __ = model.fashion_dbn_mnist.reconstruct(perturbed_data)
            if (print_reconstruction and verbose):
                print(f"\nHere's what a {num_steps}-steps reconstructed sample looks like:\n")
                plt.imshow(perturbed_data[0].detach().to('cpu').numpy().reshape(28,28))
                plt.show()
                print_reconstruction = False

        # Re-classify the perturbed image
        output = model(perturbed_data)

        # Check for success
        # get the index of the max element in the output
        final_pred = output.max(1, keepdim=True)[1]
        final_pred = output.argmax(-1)
        correct += (final_pred == target).sum()

    # Calculate final accuracy for this epsilon
    final_acc = correct/float(len(test_loader.sampler))
    print("\nEpsilon: {}\nTest Accuracy: {:.2f}%\n".format(epsilon, final_acc*100))

    return final_acc.item()

In [ ]:
#test loader is just a way to add data
test_set = torch.utils.data.TensorDataset(test_pixels, test_labels)
test_loader = torch.utils.data.DataLoader(test_set)

Let's see how good the FFNN does:

In [ ]:
final_acc = test_robustness_to_attack(ffnn, device,
                                      test_loader, epsilon=0.2,
                                      num_steps=0)


Let's now compare compare with the read-out trained on the hidden representations of the DBN.

In [ ]:
final_acc = test_robustness_to_attack(dbn_with_readout, device,
                                      test_loader, epsilon=0.2,
                                      num_steps=0)

And finally let's test whether using one step of top-down reconstruction from the generative model allows to improve resilience to attacks

In [ ]:
### TASK: repeat the same test for the DBN with 2 top-down reconstruction steps
final_acc = test_robustness_to_attack(dbn_with_readout, device,
                                      test_loader, epsilon=0.2,
                                      num_steps=2)

In [ ]:
### TASK: repeat the same test for the DBN with 2 top-down reconstruction steps
final_acc_2 = test_robustness_to_attack(dbn_with_readout, device,
                                      test_loader, epsilon=0.2,
                                      num_steps=10)

**Final accuracy on test set considering an epsilon value of 0.2:**

*   FFNN: 8.83%
*   DBN: 14.60%
*   DBN (2 steps): 17.36%
*   DBN (10 steps): 19.39%



### Effect of the noise parameter $\epsilon$

Finally, let's compare the robustness of each model to adversarial attacks of different "strengths":

In [ ]:
epsilon_values = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

def test_epsilon_values_effect(model, n_steps):
  accuracies = list()

  for eps in epsilon_values:
      acc = test_robustness_to_attack(model, device, test_loader, eps, num_steps=n_steps, verbose=False)  # set verbose to False to avoid displaying too many images
      accuracies.append(acc)

  return accuracies

In [ ]:
%%capture
accuracies_ffnn = test_epsilon_values_effect(ffnn, n_steps=0)
accuracies_dbn_0 = test_epsilon_values_effect(dbn_with_readout, n_steps=0)
accuracies_dbn_1 = test_epsilon_values_effect(dbn_with_readout, n_steps=2)

In [ ]:
accuracies_dbn_2 = test_epsilon_values_effect(dbn_with_readout, n_steps=10)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

__ = ax.axhline(0.1, color='gray', linestyle=':')
__ = ax.plot(epsilon_values, accuracies_ffnn)
__ = ax.plot(epsilon_values, accuracies_dbn_0)
__ = ax.plot(epsilon_values, accuracies_dbn_1)
__ = ax.plot(epsilon_values, accuracies_dbn_2)
__ = ax.set_xlabel("Strength of adversarial attack ($\epsilon$)")
__ = ax.set_ylabel("Accuracy")
__ = ax.set_title("Robustness to adversarial attacks", {'fontsize': 15})
__ = ax.legend(["Chance level", "FFNN", "DBN", "DBN top-down", "DBN (10 steps)"])

## Conclusion:

From these results it's clear that the FFNN model is the most vulnerable one, since it provided the lowest results even when compared to the DBN without reconstruction step. On the other hand, the reconstruction step demonstrated to improve the accuracy or help the model to "resist" the attack, and increasing the number of steps is directly related to a higher accuracy.

Nonetheless, the DBN model with 10 reconstruction steps provided the highest accuracy, the results are not satisfactory. So, when dealing with high levels of perturbation or noise, it's important to consider complementary strategies to protect our models. These strategies can include data augmentation techniques, such as, incorporating adversarial examples into the training phase.

## Reference papers
- [G. Hinton - A Practical Guide to Training Restricted Boltzmann Machines](https://www.cs.toronto.edu/~hinton/absps/guideTR.pdf)
- [G. Hinton, R. Salakhutdinov - Reducing the Dimensionality of Data with Neural Networks](https://www.science.org/doi/10.1126/science.1127647)
- [Testolin et al. - Deep unsupervised learning on a desktop PC: a primer for cognitive scientists](https://www.frontiersin.org/articles/10.3389/fpsyg.2013.00251/full)
